In [1]:
import torch, os 
from ultralytics import YOLO
from pathlib import Path

In [2]:
modelpath = Path(r"C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\YoLO.pt")
model = YOLO(modelpath)

In [3]:
model.export(
    format="ncnn",
    imgsz=320,
    device="cpu",    
)


Ultralytics 8.4.41  Python-3.11.0rc2 torch-2.11.0+cu126 CPU (13th Gen Intel Core i7-13700H)
WARNING NCNN export does not support end2end models, disabling end2end branch.


YOLO26n summary (fused): 146 layers, 2,494,694 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\YoLO.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 5, 2100) (5.1 MB)

NCNN: starting export with NCNN 1.0.20260114 and PNNX 20260409...
NCNN: export success  9.6s, saved as 'C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\YoLO_ncnn_model' (9.2 MB)

Export complete (10.2s)
Results saved to C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender
Predict:         yolo predict task=detect model=C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\YoLO_ncnn_model imgsz=320 
Validate:        yolo val task=detect model=C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\YoLO_ncnn_model imgsz=320 data=C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\Anti-UAV.yolo26\data.yaml  
Visualize:       https://netron.app


'C:\\Users\\ahmed\\Desktop\\me\\me\\Coding\\HARDCORE\\Galatic_Defender\\YoLO_ncnn_model'

In [ ]:
import glob, random, cv2, numpy as np

def make_calibration_gen(data_yaml_path, num_samples=200, imgsz=320):
    """
    Parses data.yaml to find val images and builds the TFLite
    representative dataset generator.
    """
    import yaml, os

    with open(data_yaml_path) as f:
        cfg = yaml.safe_load(f)

    # Resolve image path relative to yaml location
    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))
    val_path = os.path.join(yaml_dir, cfg.get("val", "images/val"))

    images = glob.glob(f"{val_path}/**/*.jpg", recursive=True) + \
             glob.glob(f"{val_path}/**/*.png", recursive=True)

    if len(images) == 0:
        raise ValueError(f"No images found at {val_path} — check data.yaml paths")

    random.shuffle(images)
    images = images[:num_samples]
    print(f"Calibration: using {len(images)} images from {val_path}")

    def generator():
        for img_path in images:
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (imgsz, imgsz))
            img = img.astype(np.float32) / 255.0
            yield [np.expand_dims(img, axis=0)]  # shape: (1, 320, 320, 3)

    return generator
